# LFQ Video VAE — comma2k19 training

Trains a Lookup-Free Quantization video autoencoder on comma2k19 driving footage at near-native resolution (center-cropped to 1152×864), with **16× spatial compression** and **MSE + VGG perceptual** reconstruction loss.

**Before running:** set `Runtime → Change runtime type → GPU` (A100 is best; T4 works at batch 1).

Token grid per clip: `T × 54 × 72` = ~3.9k tokens/frame at the 1152×864 crop.

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## Config

Point `DATA_ROOT` at wherever the comma2k19 chunks live in your Drive. The dataset class will recursively find all `.mkv` files.

In [ ]:
from pathlib import Path

# --- paths (EDIT THESE) ---
DATA_ROOT = Path('/content/drive/MyDrive/comma2k19')
CKPT_DIR  = Path('/content/drive/MyDrive/lfq_ckpts')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# --- input ---
CROP_W, CROP_H = 1152, 864    # center-crop (native is 1164x874)
CLIP_T  = 8                   # frames per clip
FPS     = 20                  # comma2k19 capture rate
SEG_FRAMES = 1200             # ~1 minute per .mkv

# --- model (sized for ~95 GB of GPU memory) ---
HIDDEN_DIM      = 256         # widened from 128 - more encoder capacity
EMBED_DIM       = 64
CODEBOOK_DIM    = 14          # 2^14 = 16384 codes
NUM_DOWNSAMPLES = 4           # 2^4 = 16x spatial compression

# --- loss weights ---
ENTROPY_W    = 0.1            # entropy regularizer in LFQ (paper default)
PERCEPTUAL_W = 0.1            # VGG perceptual term on top of MSE
VGG_CROP     = 384            # random crop fed to VGG; bigger = more detail signal

# --- optimization ---
# At batch=24, hidden=256, T=8 the first-conv activation is ~24 GB at fp16 and
# peak memory around the encoder/decoder boundary lands at ~70 GB. If you see OOM,
# step BATCH_SIZE down to 16; if there's lots of headroom, bump to 32.
BATCH_SIZE  = 24
GRAD_ACCUM  = 1               # no longer needed at this batch size
LR          = 5e-4            # mild bump for the larger effective batch
NUM_STEPS   = 20_000
LOG_EVERY   = 50
SAVE_EVERY  = 2_000
NUM_WORKERS = 16              # ffmpeg-decode parallelism; needs to keep up at batch 24


## Model code

Writes `lfq.py` to `/content` so we can import `LFQVAE`. Keep this cell in sync with `scripts/lfq.py` in the repo.

In [ ]:
%%writefile lfq.py
"""Lookup-Free Quantization

Self-contained autoencoder + LFQ quantizer. The model takes video segments
of shape (B, C, T, H, W) directly — no per-frame flatten needed. Encoder/decoder
use 3D convs so frames inform each other temporally; T is preserved (no
temporal compression), spatial dims are compressed 8x.

Token grid shape: (T, H/8, W/8). With T=8, H=W=64, that's (8, 8, 8) tokens
per video segment.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


def _conv3d(in_c, out_c):
    """3D conv that halves H,W and keeps T. kernel=(3,4,4), stride=(1,2,2)."""
    return nn.Conv3d(in_c, out_c, kernel_size=(3, 4, 4),
                     stride=(1, 2, 2), padding=(1, 1, 1))


def _deconv3d(in_c, out_c):
    """3D transposed conv that doubles H,W and keeps T."""
    return nn.ConvTranspose3d(in_c, out_c, kernel_size=(3, 4, 4),
                              stride=(1, 2, 2), padding=(1, 1, 1))


class Encoder(nn.Module):
    """Compresses each spatial dim by 2**num_downsamples (T preserved).

    With num_downsamples=3 and 64x64 input: -> 8x8 latent (8x compression).
    With num_downsamples=4 and 1152x864 input: -> 72x54 latent (16x compression).
    """

    def __init__(self, in_channels=3, hidden_dim=128, embed_dim=64, num_downsamples=3):
        super().__init__()
        layers = []
        in_c = in_channels
        for _ in range(num_downsamples):
            layers.append(_conv3d(in_c, hidden_dim))
            layers.append(nn.SiLU())
            in_c = hidden_dim
        layers.append(nn.Conv3d(hidden_dim, embed_dim, kernel_size=1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    """Mirror of Encoder; upsamples each spatial dim by 2**num_downsamples."""

    def __init__(self, out_channels=3, hidden_dim=128, embed_dim=64, num_downsamples=3):
        super().__init__()
        layers = [nn.Conv3d(embed_dim, hidden_dim, kernel_size=1)]
        for i in range(num_downsamples):
            layers.append(nn.SiLU())
            out_c = hidden_dim if i < num_downsamples - 1 else out_channels
            layers.append(_deconv3d(hidden_dim, out_c))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class LFQ(nn.Module):
    """Lookup-Free Quantization.

    Each of `codebook_dim` latent dimensions is binarized to {-1, +1}.
    The implicit codebook is the full {-1, +1}^codebook_dim hypercube
    (size 2**codebook_dim), so there is no embedding table to look up.

    Loss = commitment_loss_weight * ||z - sign(z).detach()||^2
         + entropy_loss_weight   * (E[H(q|z)] - gamma * H(E[q|z]))

    where q(c|z) ∝ exp(2 z·c / temperature). The entropy term pulls per-sample
    assignments toward one-hot (low E[H]) while spreading usage across the
    codebook (high H(E)).

    Input shape: (B, D, *spatial), where *spatial is any number of trailing
    dims (e.g. (T, H, W) for video, (H, W) for images).
    """

    def __init__(
        self,
        embed_dim: int,
        codebook_dim: int = 14,
        entropy_loss_weight: float = 0.1,
        commitment_loss_weight: float = 0.25,
        diversity_gamma: float = 1.0,
        temperature: float = 1.0,
        usage_ema_decay: float = 0.99,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.codebook_dim = codebook_dim
        self.codebook_size = 2 ** codebook_dim
        self.entropy_loss_weight = entropy_loss_weight
        self.commitment_loss_weight = commitment_loss_weight
        self.diversity_gamma = diversity_gamma
        self.temperature = temperature
        self.usage_ema_decay = usage_ema_decay

        if embed_dim == codebook_dim:
            self.project_in = nn.Identity()
            self.project_out = nn.Identity()
        else:
            self.project_in = nn.Linear(embed_dim, codebook_dim)
            self.project_out = nn.Linear(codebook_dim, embed_dim)

        # bit weights for converting a {0,1}^D binary code into an int token id
        bits_to_int = 2 ** torch.arange(codebook_dim - 1, -1, -1, dtype=torch.long)
        self.register_buffer("bits_to_int", bits_to_int, persistent=False)

        # Explicit enumeration of all 2^D codes in {-1, +1}^D for the entropy loss.
        # Fine up to ~2^16; above that, switch to the per-dim factorized approximation.
        ids = torch.arange(self.codebook_size, dtype=torch.long)
        bits = ((ids.unsqueeze(-1) & bits_to_int) > 0).float()
        self.register_buffer("codebook", bits * 2 - 1, persistent=False)

        # EMA usage counter, exposed via codebook_usage().
        self.register_buffer("code_usage", torch.zeros(self.codebook_size))

    def forward(self, z):
        # z: [B, embed_dim, *spatial]
        B = z.shape[0]
        spatial = z.shape[2:]

        # [B, embed_dim, *spatial] -> [B, *spatial, embed_dim] -> [N, embed_dim]
        z = z.movedim(1, -1).contiguous()
        flat_z = z.reshape(-1, self.embed_dim)
        flat_z = self.project_in(flat_z)  # [N, codebook_dim]

        # Independent per-dim binarization. sign(0) -> +1.
        quantized = torch.where(
            flat_z >= 0, flat_z.new_ones(()), -flat_z.new_ones(())
        )

        bits = (quantized > 0).long()
        token_ids = (bits * self.bits_to_int).sum(dim=-1)  # [N]

        # --- commitment loss ---
        commitment_loss = F.mse_loss(flat_z, quantized.detach())

        # --- entropy loss ---
        # For ±1 codes, ||c||^2 = D is constant, so softmax(-||z-c||^2/τ)
        # reduces to softmax(2 z·c / τ).
        logits = (2.0 / self.temperature) * (flat_z @ self.codebook.t())  # [N, 2^D]
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)

        per_sample_entropy = -(probs * log_probs).sum(dim=-1).mean()
        avg_probs = probs.mean(dim=0)
        batch_entropy = -(avg_probs * (avg_probs + 1e-10).log()).sum()
        entropy_loss = per_sample_entropy - self.diversity_gamma * batch_entropy

        loss = (
            self.commitment_loss_weight * commitment_loss
            + self.entropy_loss_weight * entropy_loss
        )

        # --- straight-through estimator ---
        quantized_st = flat_z + (quantized - flat_z).detach()
        quantized_st = self.project_out(quantized_st)  # [N, embed_dim]
        quantized_st = quantized_st.view(B, *spatial, self.embed_dim).movedim(-1, 1).contiguous()

        if self.training:
            with torch.no_grad():
                batch_usage = torch.bincount(
                    token_ids, minlength=self.codebook_size
                ).to(self.code_usage.dtype)
                self.code_usage.mul_(self.usage_ema_decay).add_(
                    batch_usage, alpha=1 - self.usage_ema_decay
                )

        token_ids = token_ids.view(B, *spatial)
        return quantized_st, loss, token_ids

    def codebook_usage(self):
        used = (self.code_usage > 1e-3).sum().item()
        return used, self.codebook_size

    def reset_dead_entries(self, *args, **kwargs):
        # No-op for parity with VectorQuantizer's interface. Entropy loss
        # already pushes the model to use every code.
        return 0

    def decode_from_ids(self, token_ids):
        """[B, *spatial] long token ids -> [B, embed_dim, *spatial] features."""
        bits = (token_ids.unsqueeze(-1) & self.bits_to_int) > 0
        codes = bits.to(self.codebook.dtype) * 2 - 1
        out = self.project_out(codes)  # [B, *spatial, embed_dim]
        return out.movedim(-1, 1).contiguous()


class LFQVAE(nn.Module):
    """Video autoencoder with LFQ. Takes (B, C, T, H, W) directly."""

    def __init__(
        self,
        in_channels: int = 3,
        hidden_dim: int = 128,
        embed_dim: int = 64,
        codebook_dim: int = 14,
        num_downsamples: int = 3,
        entropy_loss_weight: float = 0.1,
        commitment_loss_weight: float = 0.25,
        # accepted but unused; lets train.py's --num-embeddings flag pass through
        num_embeddings: int | None = None,
    ):
        super().__init__()
        self.encoder = Encoder(in_channels, hidden_dim, embed_dim, num_downsamples)
        self.quantizer = LFQ(
            embed_dim=embed_dim,
            codebook_dim=codebook_dim,
            entropy_loss_weight=entropy_loss_weight,
            commitment_loss_weight=commitment_loss_weight,
        )
        self.decoder = Decoder(in_channels, hidden_dim, embed_dim, num_downsamples)

    def forward(self, x):
        # x: (B, C, T, H, W)
        z = self.encoder(x)
        q, q_loss, tokens = self.quantizer(z)
        recon = self.decoder(q)
        return recon, q_loss, tokens

    def encode(self, x):
        z = self.encoder(x)
        _, _, tokens = self.quantizer(z)
        return tokens  # (B, T, H/8, W/8)

    def decode_from_tokens(self, token_ids):
        # token_ids: (B, T, H/8, W/8)
        q = self.quantizer.decode_from_ids(token_ids)
        return self.decoder(q)


if __name__ == "__main__":
    model = LFQVAE(codebook_dim=14)
    x = torch.randn(2, 3, 8, 64, 64)  # B=2, C=3, T=8, H=W=64
    recon, loss, tokens = model(x)
    print(f"recon:  {tuple(recon.shape)}")
    print(f"tokens: {tuple(tokens.shape)}  range [{tokens.min().item()}, {tokens.max().item()}]")
    print(f"loss:   {loss.item():.4f}")
    print(f"codebook size: {model.quantizer.codebook_size}")
    print(f"params: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from lfq import LFQVAE

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

model = LFQVAE(
    hidden_dim=HIDDEN_DIM,
    embed_dim=EMBED_DIM,
    codebook_dim=CODEBOOK_DIM,
    num_downsamples=NUM_DOWNSAMPLES,
    entropy_loss_weight=ENTROPY_W,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
lat_h = CROP_H // (2 ** NUM_DOWNSAMPLES)
lat_w = CROP_W // (2 ** NUM_DOWNSAMPLES)
print(f'params: {n_params:,}')
print(f'codebook: 2^{CODEBOOK_DIM} = {2**CODEBOOK_DIM:,} codes')
print(f'tokens per clip: {CLIP_T} x {lat_h} x {lat_w} = {CLIP_T * lat_h * lat_w:,}')

## Dataset

Streams T-frame clips from random `.mkv` segments via `ffmpeg -ss`. Each DataLoader worker spawns its own ffmpeg subprocess, so I/O parallelism scales with `NUM_WORKERS`. Failed reads (corrupt segment, short file) silently skip to the next file.

In [ ]:
import subprocess
import random
import numpy as np
from torch.utils.data import IterableDataset, DataLoader


class CommaClipDataset(IterableDataset):
    """Yields (C, T, H, W) clips in [0, 1]."""

    def __init__(self, mkv_paths, clip_length=8, crop_h=864, crop_w=1152,
                 fps=20, seg_frames=1200):
        self.mkvs = [str(p) for p in mkv_paths]
        self.clip_length = clip_length
        self.crop_h = crop_h
        self.crop_w = crop_w
        self.fps = fps
        self.seg_frames = seg_frames

    def _read_clip(self, mkv_path, start_frame):
        start_time = start_frame / self.fps
        cmd = [
            'ffmpeg', '-hide_banner', '-loglevel', 'error',
            '-ss', f'{start_time:.3f}',
            '-i', mkv_path,
            '-frames:v', str(self.clip_length),
            '-vf', f'crop={self.crop_w}:{self.crop_h}',
            '-pix_fmt', 'rgb24',
            '-f', 'rawvideo', 'pipe:1',
        ]
        try:
            raw = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, timeout=30)
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
            return None
        arr = np.frombuffer(raw, dtype=np.uint8).copy()
        try:
            arr = arr.reshape(-1, self.crop_h, self.crop_w, 3)
        except ValueError:
            return None
        if arr.shape[0] != self.clip_length:
            return None
        return arr

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        seed = info.seed if info is not None else 0
        rng = random.Random(seed)
        while True:
            mkv = rng.choice(self.mkvs)
            start = rng.randint(0, max(0, self.seg_frames - self.clip_length))
            arr = self._read_clip(mkv, start)
            if arr is None:
                continue
            x = torch.from_numpy(arr).float() / 255.0      # (T, H, W, C)
            yield x.permute(3, 0, 1, 2).contiguous()       # (C, T, H, W)


# discover all .mkv segments under DATA_ROOT
mkv_paths = sorted(DATA_ROOT.rglob('*.mkv'))
assert mkv_paths, f'no .mkv files under {DATA_ROOT} - check the path'
print(f'found {len(mkv_paths)} .mkv segments')

# small holdout for visualization
random.seed(0)
random.shuffle(mkv_paths)
val_paths   = mkv_paths[:max(1, len(mkv_paths) // 50)]
train_paths = mkv_paths[len(val_paths):]
print(f'train: {len(train_paths)} segments | val: {len(val_paths)}')

train_ds = CommaClipDataset(train_paths, CLIP_T, CROP_H, CROP_W, FPS, SEG_FRAMES)
val_ds   = CommaClipDataset(val_paths,   CLIP_T, CROP_H, CROP_W, FPS, SEG_FRAMES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=1, pin_memory=True)

## Perceptual loss

VGG16 features at relu1_2 / relu2_2 / relu3_3 — pulls detail back into the reconstructions that pure MSE would blur away.

VGG is applied per-frame on a random 256×256 crop (same crop for `x` and `recon`). The full-resolution VGG forward at 1152×864 would be ~2 GB of activations per layer; the random crop covers the whole image stochastically across training while keeping the memory footprint flat.

In [ ]:
from torchvision.models import vgg16, VGG16_Weights


class VGGPerceptualLoss(nn.Module):
    def __init__(self, layer_indices=(3, 8, 15), crop_size=VGG_CROP):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.DEFAULT).features.eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.layer_indices = set(layer_indices)
        self.max_layer = max(layer_indices)
        self.crop_size = crop_size
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, y):
        # x, y: (B, C, T, H, W) in [0, 1]
        B, C, T, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
        y = y.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)

        cs = self.crop_size
        if H > cs and W > cs:
            top  = torch.randint(0, H - cs + 1, (1,)).item()
            left = torch.randint(0, W - cs + 1, (1,)).item()
            x = x[:, :, top:top + cs, left:left + cs]
            y = y[:, :, top:top + cs, left:left + cs]

        x = (x.clamp(0, 1) - self.mean) / self.std
        y = (y.clamp(0, 1) - self.mean) / self.std

        loss = 0.0
        for i, layer in enumerate(self.vgg):
            x = layer(x)
            y = layer(y)
            if i in self.layer_indices:
                loss = loss + F.mse_loss(x, y)
            if i >= self.max_layer:
                break
        return loss


perceptual = VGGPerceptualLoss().to(device)

## Train

Mixed-precision (fp16) + gradient accumulation. Logs codebook utilization alongside losses — watch this to catch collapse early. If `used / 16384` stays below ~1% after the first 1k steps, bump `ENTROPY_W` to 0.3 or 1.0 and restart.

Auto-resumes from the latest checkpoint in `CKPT_DIR` if you re-run the cell.

In [ ]:
import time

opt = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()

# resume support
start_step = 0
ckpts = sorted(CKPT_DIR.glob('ckpt_*.pt'))
if ckpts:
    state = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(state['model'])
    opt.load_state_dict(state['opt'])
    start_step = state['step']
    print(f'resumed from {ckpts[-1].name} at step {start_step}')
else:
    print('no checkpoint found, starting fresh')

loader_iter = iter(train_loader)
log = {'step': [], 'recon': [], 'perc': [], 'q': [], 'used': []}

model.train()
t0 = time.time()
step = start_step
while step < NUM_STEPS:
    opt.zero_grad(set_to_none=True)
    recon_acc = perc_acc = q_acc = 0.0
    for _ in range(GRAD_ACCUM):
        batch = next(loader_iter).to(device, non_blocking=True)
        with torch.cuda.amp.autocast(dtype=torch.float16):
            recon, q_loss, _ = model(batch)
            mse  = F.mse_loss(recon, batch)
            perc = perceptual(recon, batch)
            total = (mse + PERCEPTUAL_W * perc + q_loss) / GRAD_ACCUM
        scaler.scale(total).backward()
        recon_acc += mse.item()
        perc_acc  += perc.item()
        q_acc     += q_loss.item()
    scaler.step(opt)
    scaler.update()

    if step % LOG_EVERY == 0:
        used, total_codes = model.quantizer.codebook_usage()
        dt = time.time() - t0
        sps = (step - start_step + 1) / max(dt, 1e-6)
        print(
            f'step {step:>6d} | '
            f'recon {recon_acc/GRAD_ACCUM:.4f} | '
            f'perc {perc_acc/GRAD_ACCUM:.4f} | '
            f'q {q_acc/GRAD_ACCUM:.4f} | '
            f'codebook {used}/{total_codes} ({100*used/total_codes:.1f}%) | '
            f'{sps:.2f} steps/s'
        )
        log['step'].append(step)
        log['recon'].append(recon_acc/GRAD_ACCUM)
        log['perc'].append(perc_acc/GRAD_ACCUM)
        log['q'].append(q_acc/GRAD_ACCUM)
        log['used'].append(used)

    if (step + 1) % SAVE_EVERY == 0:
        ckpt_path = CKPT_DIR / f'ckpt_{step+1:06d}.pt'
        torch.save({
            'step': step + 1,
            'model': model.state_dict(),
            'opt': opt.state_dict(),
            'config': dict(hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM,
                           codebook_dim=CODEBOOK_DIM, num_downsamples=NUM_DOWNSAMPLES),
        }, ckpt_path)
        print(f'  saved {ckpt_path.name}')

    step += 1

print(f'\ndone in {(time.time() - t0) / 60:.1f} min')

## Visualize reconstructions

Pulls one val clip, runs forward, and shows original vs reconstruction across the T frames. Crops a 512×512 center patch for legibility — the full 1152×864 strip would be too wide for a notebook display.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model.eval()
batch = next(iter(val_loader)).to(device)
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=torch.float16):
        recon, _, tokens = model(batch)

orig = batch[0].clamp(0, 1).float().cpu()
rec  = recon[0].clamp(0, 1).float().cpu()

# crop a center 512x512 patch for display
ch, cw = 512, 512
h0 = (orig.shape[2] - ch) // 2
w0 = (orig.shape[3] - cw) // 2
orig = orig[:, :, h0:h0+ch, w0:w0+cw]
rec  = rec[:,  :, h0:h0+ch, w0:w0+cw]

T = orig.shape[1]
fig, axes = plt.subplots(2, T, figsize=(T * 2.2, 4.4))
for t in range(T):
    axes[0, t].imshow(orig[:, t].permute(1, 2, 0).numpy())
    axes[0, t].axis('off')
    axes[1, t].imshow(rec[:, t].permute(1, 2, 0).numpy())
    axes[1, t].axis('off')
axes[0, 0].set_title('orig', loc='left')
axes[1, 0].set_title('recon', loc='left')
plt.tight_layout()
plt.show()

mse = F.mse_loss(recon.float(), batch.float()).item()
psnr = 10 * np.log10(1.0 / max(mse, 1e-10))
print(f'val mse: {mse:.5f}  psnr: {psnr:.2f} dB  tokens: {tuple(tokens.shape)}')